# Notebook 04 — Model C Production Runner

## D-083 — Model C

This notebook is a standalone production runner for **Model C (33,497,600 parameters)**.

It can start from a fresh Colab T4 runtime. It clones/pulls the repository, mounts the persistent Google Drive production directory, imports the canonical `src/training_pipeline.py`, reconstructs or loads the pinned canonical corpus, and then trains/resumes/verifies Model C.

It does **not** rerun Models A or B, the learning-rate probes, or the accelerator preflight.

Frozen production controls: 20M-token training corpus, context 512, peak LR `2e-3`, AdamW, effective batch 16,384 targets/update, physical micro-batch 32 sequences on T4, 3 epochs / 3,663 updates, validation every 200 updates plus epoch end, seed 42, FP16 + GradScaler, official test split untouched.

If a complete persistent Model C run already exists, it is verified and reused. If an incomplete resumable v2 checkpoint exists, training resumes exactly from its most recent persisted validation boundary.

Canonical decision history: `docs/decisions/04_training_pipeline.md`; project-wide index: `docs/DECISION_INDEX.md`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

COLAB_ROOT = Path("/content")
REPO = COLAB_ROOT / "foundation-model-from-scratch"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/traderjohnd/foundation-model-from-scratch.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "origin", "main"],
        check=True,
    )

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

def ensure_package(import_name: str, install_spec: str):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", install_spec],
            check=True,
        )
        return importlib.import_module(import_name)

ensure_package("datasets", "datasets")
ensure_package("tokenizers", "tokenizers==0.23.1")

assert Path("src/model.py").exists()
assert Path("src/data.py").exists()
assert Path("src/training_pipeline.py").exists()
assert Path("results/tokenizer/tokenizer.json").exists()

print("Repository/bootstrap audit: PASS")
print("Working directory:", os.getcwd())


In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")

try:
    if not (DRIVE_ROOT / "MyDrive").exists():
        drive.mount(str(DRIVE_ROOT), force_remount=False)
except Exception:
    drive.mount(str(DRIVE_ROOT), force_remount=True)

PERSISTENT_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
    / "foundation-model-from-scratch"
    / "production"
)
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

print("Persistent production root:")
print(PERSISTENT_ROOT)


In [ ]:
from src.training_pipeline import (
    audit_persistent_model_artifacts,
    run_production_model,
)

model_c_summary = run_production_model(
    "C",
    PERSISTENT_ROOT,
    require_t4=True,
)

audit_c = audit_persistent_model_artifacts(
    "C",
    PERSISTENT_ROOT,
)

assert model_c_summary["completed_updates"] == 3_663
assert model_c_summary["completed_full_epochs"] == 3
assert model_c_summary["total_target_exposures"] == 59_999_232
assert model_c_summary["validation_events"] == 21
assert model_c_summary["official_test_split_content_used"] is False

print()
print("MODEL C PERSISTENT ARTIFACT AUDIT: PASS")
print("Best validation loss:", f'{model_c_summary["best_validation_loss"]:.6f}')
print("Best perplexity:", f'{model_c_summary["best_validation_perplexity"]:.2f}')
print("Best update:", model_c_summary["best_validation_update"])
print("Official test split used: NO")
